In [12]:
DATA = "bookings.jsonl"

# Task 2: Contracts, nested data and a star schema
Setup: imports and output paths.

In [14]:
import json
from collections import Counter
from datetime import date, datetime
from pathlib import Path
from typing import Literal, Optional

import duckdb
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, model_validator

QUARANTINE = "quarantine.jsonl"
STAR = Path("star")

## Part A: The contract
### A1. Pydantic models

In [15]:
Faculty = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
Club = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TicketType = Literal["standard", "vip", "student"]
Payment = Literal["card", "cash", "wallet"]
Status = Literal["confirmed", "cancelled"]


class Ticket(BaseModel):
    type: TicketType
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    id: str = Field(pattern=r"^\d{2}-\d{4}$")
    name: str
    faculty: Faculty


class Event(BaseModel):
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    club: Club
    date: date
    venue: str = Field(min_length=1)


class Booking(BaseModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event                      # required: null or missing fails
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: Payment
    status: Status
    promo_code: Optional[str] = None  # drifted field: absent in older documents -> None

    @model_validator(mode="after")
    def total_matches_tickets(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if abs(self.total - expected) > 0.005:
            raise ValueError("total does not equal sum of qty x price")
        return self

    @model_validator(mode="after")
    def booked_before_event(self):
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked on or after the event date")
        return self

### A2 + A3. Validate every line, then drop re-sent bookings
Validation runs first. Only bookings that passed are checked for duplicates, and the first valid copy of each `booking_id` is kept.

In [16]:
def short_reason(err: ValidationError) -> str:
    """One readable reason per booking: the first error, as 'field: message'."""
    e = err.errors()[0]
    # list positions become [] so "tickets.0.qty" and "tickets.1.qty" count as one reason
    loc = ".".join("[]" if isinstance(p, int) else str(p) for p in e["loc"]).replace(".[]", "[]") or "booking"
    msg = e["msg"].removeprefix("Value error, ")
    return f"{loc}: {msg}"


lines_read = 0
valid, quarantine = [], []

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        lines_read += 1
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"line": line_no, "booking_id": None, "reason": "invalid JSON"})
            continue
        bid = doc.get("booking_id") if isinstance(doc, dict) else None
        try:
            valid.append(Booking.model_validate(doc))
        except ValidationError as err:
            quarantine.append({"line": line_no, "booking_id": bid, "reason": short_reason(err)})

# A3: duplicates, among valid bookings only
seen, kept = set(), []
for b in valid:
    if b.booking_id in seen:
        quarantine.append({"booking_id": b.booking_id, "reason": "duplicate booking_id"})
    else:
        seen.add(b.booking_id)
        kept.append(b)
valid = kept

with open(QUARANTINE, "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q) + "\n")

### A4. Report

In [17]:
print(f"Lines read:   {lines_read}")
print(f"Passed:       {len(valid)}")
print(f"Quarantined:  {len(quarantine)}")
assert lines_read == len(valid) + len(quarantine)

reasons = pd.Series([q["reason"] for q in quarantine]).value_counts().rename_axis("reason").reset_index(name="count")
reasons

Lines read:   283
Passed:       259
Quarantined:  24


,reason,count
0,student.id: String should match pattern '^\d{2...,4
1,booking: total does not equal sum of qty x price,4
2,"tickets[].type: Input should be 'standard', 'v...",3
3,tickets[].qty: Input should be greater than or...,3
4,booking: booked on or after the event date,3
5,duplicate booking_id,3
6,event: Input should be a valid dictionary or i...,2
7,tickets: List should have at least 1 item afte...,2


### A5. What if the duplicate check ran before validation?
For Part D: rerun the same pipeline in the other order and compare the counts.

In [18]:
seen_raw, dedup_first_q, dedup_first_pass = set(), Counter(), 0
with open(DATA, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            doc = json.loads(line)
        except json.JSONDecodeError:
            dedup_first_q["invalid JSON"] += 1
            continue
        bid = doc.get("booking_id") if isinstance(doc, dict) else None
        if bid in seen_raw:
            dedup_first_q["duplicate booking_id"] += 1
            continue
        seen_raw.add(bid)
        try:
            Booking.model_validate(doc)
            dedup_first_pass += 1
        except ValidationError as err:
            dedup_first_q[short_reason(err)] += 1

swap = reasons.set_index("reason")["count"].rename("validate_then_dedup").to_frame().join(
    pd.Series(dedup_first_q, name="dedup_then_validate"), how="outer").fillna(0).astype(int)
print("passed: validate->dedup =", len(valid), "| dedup->validate =", dedup_first_pass)
swap

passed: validate->dedup = 259 | dedup->validate = 259


,validate_then_dedup,dedup_then_validate
reason,,
booking: booked on or after the event date,3,3
booking: total does not equal sum of qty x price,4,4
duplicate booking_id,3,3
event: Input should be a valid dictionary or instance of Event,2,2
student.id: String should match pattern '^\d{2}-\d{4}$',4,4
"tickets: List should have at least 1 item after validation, not 0",2,2
tickets[].qty: Input should be greater than or equal to 1,3,3
"tickets[].type: Input should be 'standard', 'vip' or 'student'",3,3


## Part B: Flatten
### B1. `bookings` (one row per booking) and `booking_tickets` (one row per ticket entry)

In [19]:
docs = [b.model_dump(mode="json") for b in valid]

bookings = pd.json_normalize(docs, sep="_").drop(columns=["tickets"])
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True).dt.tz_convert(None)  # naive UTC
bookings["event_date"] = pd.to_datetime(bookings["event_date"])

booking_tickets = pd.json_normalize(docs, record_path="tickets", meta=["booking_id"])
booking_tickets = booking_tickets[["booking_id", "type", "qty", "price"]]
booking_tickets["amount"] = booking_tickets["qty"] * booking_tickets["price"]

print(bookings.shape, booking_tickets.shape)
display(bookings.head())
display(booking_tickets.head())

(259, 14) (314, 5)


,booking_id,booked_at,total,payment,status,promo_code,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue
0,B0001,2026-09-28 15:46:00,340.0,card,cancelled,NaN,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,B0002,2026-09-27 12:29:00,40.0,card,confirmed,NaN,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
2,B0003,2026-10-13 20:19:00,60.0,wallet,confirmed,NaN,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
3,B0004,2026-11-01 06:42:00,105.0,card,confirmed,CLUB10,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
4,B0005,2026-10-28 19:06:00,20.0,cash,confirmed,NaN,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


### B2. Assertions

In [20]:
assert bookings["booking_id"].is_unique, "booking_id is not unique in bookings"

per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].to_frame().join(per_booking)
assert check["amount"].notna().all(), "a booking has no tickets"
assert ((check["amount"] - check["total"]).abs() < 0.005).all(), "ticket amounts don't add up to total"
print("B2 assertions passed")

B2 assertions passed


### B3. Schema drift
Compare every raw document's field paths against the fields in the README example.

In [21]:
EXPECTED = {
    "booking_id", "booked_at", "total", "payment", "status", "student", "event", "tickets",
    "student.id", "student.name", "student.faculty",
    "event.id", "event.title", "event.club", "event.date", "event.venue",
    "tickets[].type", "tickets[].qty", "tickets[].price",
}

def paths(doc):
    out = set(doc)
    for k in ("student", "event"):
        if isinstance(doc.get(k), dict):
            out |= {f"{k}.{c}" for c in doc[k]}
    for t in doc.get("tickets") or []:
        if isinstance(t, dict):
            out |= {f"tickets[].{c}" for c in t}
    return out

raw = []
with open(DATA, encoding="utf-8") as f:
    for line in f:
        try:
            raw.append(json.loads(line))
        except json.JSONDecodeError:
            pass

drift = Counter(p for d in raw for p in paths(d) - EXPECTED)
print("Fields not in the example:", dict(drift))

for field in drift:
    present = [d for d in raw if field in paths(d)]
    absent = [d for d in raw if field not in paths(d)]
    first = min(d["booked_at"] for d in present)
    print(f"\n'{field}': in {len(present)} docs, missing from {len(absent)}")
    print(f"  first appears: booked_at {first}")
    print(f"  latest doc without it: booked_at {max(d['booked_at'] for d in absent)}")
    # same check on valid bookings only
    if field in bookings.columns:
        v_has = bookings[[field in json.loads(b.model_dump_json(exclude_unset=True)) for b in valid]]
        v_not = bookings.drop(v_has.index)
        print(f"  valid bookings with it: {len(v_has)}, earliest booked_at {v_has['booked_at'].min()}")
        print(f"  valid bookings without it: {len(v_not)}, latest booked_at {v_not['booked_at'].max()}")
        print(f"  in the bookings table it is a column; missing values are null: {bookings[field].isna().sum()} nulls")

Fields not in the example: {'promo_code': 193}

'promo_code': in 193 docs, missing from 90
  first appears: booked_at 2026-10-20T01:05:00Z
  latest doc without it: booked_at 2026-10-31T10:00:00Z
  valid bookings with it: 173, earliest booked_at 2026-10-20 01:05:00
  valid bookings without it: 86, latest booked_at 2026-10-19 18:43:00
  in the bookings table it is a column; missing values are null: 172 nulls


## Part C: Model and store
### C1. Star schema in DuckDB (fact grain: one ticket entry)

In [22]:
con = duckdb.connect()   # in-memory database
con.register("bookings_df", bookings)
con.register("tickets_df", booking_tickets)

# one row per event: if an id ever comes with different attributes, keep its earliest booking's version
con.execute("""
CREATE OR REPLACE TABLE dim_event AS
SELECT ROW_NUMBER() OVER (ORDER BY event_id) AS event_key, *
FROM (
    SELECT event_id, event_title, event_club, CAST(event_date AS DATE) AS event_date, event_venue
    FROM bookings_df
    QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY booked_at, booking_id) = 1
)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_student AS
SELECT ROW_NUMBER() OVER (ORDER BY student_id) AS student_key, *
FROM (
    SELECT student_id, student_name, student_faculty
    FROM bookings_df
    QUALIFY ROW_NUMBER() OVER (PARTITION BY student_id ORDER BY booked_at, booking_id) = 1
)
""")

con.execute("""
CREATE OR REPLACE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key,
       t.type   AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status
FROM tickets_df  t
JOIN bookings_df b USING (booking_id)
JOIN dim_event   e ON e.event_id   = b.event_id
JOIN dim_student s ON s.student_id = b.student_id
""")

for t in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
con.execute("SELECT * FROM fact_ticket_sales LIMIT 5").df()

dim_event 8
dim_student 103
fact_ticket_sales 314


,booking_id,event_key,student_key,date_key,ticket_type,qty,price,amount,status
0,B0001,1,61,20260928,standard,1,40.0,40.0,cancelled
1,B0001,1,61,20260928,vip,3,100.0,300.0,cancelled
2,B0002,1,84,20260927,standard,1,40.0,40.0,confirmed
3,B0003,2,14,20261013,standard,2,30.0,60.0,confirmed
4,B0004,7,15,20261101,standard,3,35.0,105.0,confirmed


### C2. Assertions

In [23]:
fact_rows = con.execute("SELECT COUNT(*) FROM fact_ticket_sales").fetchone()[0]
assert fact_rows == len(booking_tickets), (fact_rows, len(booking_tickets))

fact_amount = con.execute("SELECT SUM(amount) FROM fact_ticket_sales").fetchone()[0]
assert abs(fact_amount - bookings["total"].sum()) < 0.01, (fact_amount, bookings["total"].sum())

# every fact row found its dimensions
assert con.execute("SELECT COUNT(*) FROM fact_ticket_sales WHERE event_key IS NULL OR student_key IS NULL").fetchone()[0] == 0
print(f"C2 passed: {fact_rows} fact rows, SUM(amount) = {fact_amount:.2f} = SUM(total)")

C2 passed: 314 fact rows, SUM(amount) = 31285.00 = SUM(total)


### C3. Confirmed bookings: tickets sold and revenue per club and ticket type

In [24]:
answer = con.execute("""
SELECT e.event_club     AS club,
       f.ticket_type,
       SUM(f.qty)       AS tickets_sold,
       SUM(f.amount)    AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e USING (event_key)
WHERE f.status = 'confirmed'
GROUP BY ALL
ORDER BY club, ticket_type
""").df()
answer

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


### C4. Save as Parquet in `star/`

In [25]:
import shutil
shutil.rmtree(STAR, ignore_errors=True)
STAR.mkdir()

con.execute(f"COPY dim_event   TO '{STAR}/dim_event.parquet'   (FORMAT PARQUET)")
con.execute(f"COPY dim_student TO '{STAR}/dim_student.parquet' (FORMAT PARQUET)")
con.execute(f"""
COPY (SELECT f.*, strftime(b.booked_at, '%Y-%m') AS booked_month
      FROM fact_ticket_sales f
      JOIN bookings_df b USING (booking_id))
TO '{STAR}/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

for p in sorted(STAR.rglob("*.parquet")):
    print(p)

# read back to prove the files are complete
back = con.execute(f"SELECT COUNT(*), SUM(amount) FROM read_parquet('{STAR}/fact_ticket_sales/*/*.parquet', hive_partitioning = true)").fetchone()
assert back[0] == fact_rows
print("Read back from Parquet:", back)

star\dim_event.parquet
star\dim_student.parquet
star\fact_ticket_sales\booked_month=2026-09\data_0.parquet
star\fact_ticket_sales\booked_month=2026-10\data_0.parquet
star\fact_ticket_sales\booked_month=2026-11\data_0.parquet
Read back from Parquet: (314, 31285.0)
